In [ ]:
from pathlib import Path

from skald import combine_runs
import seaborn as sns
import polars as pl

import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator
import numpy as np

from experiments.visualizations import set_paper_style

set_paper_style()

In [ ]:
runs = combine_runs(
    directory=Path(r"W:\EndlessWM\experiments\reports\streams"),
    include_params=["data.name", "data.dataset_seed", "method.name", "seed", "skald.run_name"],
)
completed_runs = runs.filter(pl.col("name") == "total-runtime")["skald.run_name"].unique()
metrics = runs.filter(
    pl.col("skald.run_name").is_in(completed_runs),
    pl.col("epoch").is_null(),
    pl.col("stream").is_in(["I", "II", "III", "IV"]),
    name="mAP",
    pattern="*",
)
metrics

In [ ]:
fig = sns.relplot(
    metrics,
    x="task",
    y="value",
    hue="stream",
    col="method.name",
    row="data.name",
    kind="line",
    legend="full",
    facet_kws={"sharex": False},
)
fig

In [ ]:
summary = (
    metrics.filter(
        pl.col("epoch").is_null(),
        name="mAP",
        pattern="*",
    )
    # Aggregates over stream, data.dataset_seed, and task
    .group_by(["data.name", "method.name"])
    .agg(
        pl.col("value").mean().alias("mean_map"),
        (1.96 * pl.col("value").std(ddof=1) / pl.len().sqrt()).alias("confidence"),
    )
    .with_columns(
        pl.concat_str(
            [
                pl.col("mean_map").round(4).cast(pl.String),
                pl.lit("±"),
                pl.col("confidence").round(4).cast(pl.String),
            ]
        ).alias("mean_map_ci")
    )
    .sort(["data.name", "method.name"])
)

_method_order = ["finetuning", "replay", "joint", "ewc", "lwf", "path_integral", "pirb", "pass", "remind", "siesta"]

_pivot = (
    summary.pivot(
        on="data.name",
        index="method.name",
        values="mean_map_ci",
    )
    .select(["method.name", "i-e", "ii-e", "iii-e", "iv-e", "i-r", "ii-r", "iii-r", "iv-r"])
    .with_columns(pl.col("method.name").cast(pl.Enum(_method_order)))
    .sort("method.name")
)

display(_pivot)
print(_pivot.to_pandas().to_latex(index=False))

In [ ]:
set_paper_style()
plt.rcParams.update(
    {
        "figure.figsize": (3.5, 1.95),
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "lines.linewidth": 0.9,
        "lines.markersize": 3,
    },
)

_stream_order = ["I", "II", "III", "IV"]

_plot_df = metrics.filter(
    pl.col("data.name").is_in(["ii-e", "ii-r"]),
    pl.col("stream").is_in(_stream_order),
    pl.col("epoch").is_null(),
    pl.col("pattern") == "*",
    pl.col("method.name").is_in(["finetuning", "replay"]),
).rename({"method.name": "method"})

_output_dir = Path("../../reports/cl")
_output_dir.mkdir(parents=True, exist_ok=True)

for _dataset, _stem, _show_legend in [
    ("ii-e", "cl_results_ii_e", True),
    ("ii-r", "cl_results_ii_r", False),
]:
    _grid = sns.relplot(
        data=_plot_df.filter(pl.col("data.name") == _dataset),
        x="task",
        y="value",
        hue="stream",
        hue_order=_stream_order,
        col="method",
        kind="line",
        height=1.6,
        aspect=1.3,
        facet_kws={"sharex": False, "sharey": True},
        legend="full" if _show_legend else False,
    )

    _grid.set_axis_labels("task $t$", "mAP")
    _grid.set_titles(col_template="{col_name}")

    for _ax in _grid.axes.flat:
        _ax.set_ylim(0.0, 1.0)
        _ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
        _ax.grid(True, axis="y", alpha=0.22, linewidth=0.5)

    if _show_legend:
        sns.move_legend(
            _grid,
            "lower center",
            bbox_to_anchor=(0.56, 0.9),
            ncol=len(_stream_order),
            title=None,
            frameon=True,
            borderaxespad=1.0,
            borderpad=0.25,
            handlelength=1.0,
            handletextpad=0.5,
            columnspacing=1.0,
            labelspacing=0.2,
        )

    _grid.fig.tight_layout()

    _grid.fig.savefig(_output_dir / f"{_stem}.png", bbox_inches="tight", dpi=300)
    _grid.fig.savefig(_output_dir / f"{_stem}.pdf", bbox_inches="tight")

    display(_grid.fig)